# 02 — Teacher fine-tune (`xlm-roberta-base`)

Fine-tunes the full `xlm-roberta-base` teacher on the English Jigsaw training set prepared by `training/01_data_prep.py`, using class-weighted multi-label BCE. This checkpoint's soft-label logits feed the student distillation notebook (`03_student_distill`, not yet built).

**Runtime:** Colab Pro, GPU runtime set to A100 or H100.

## Setup

In [ ]:
# Public repo, no auth needed
!git clone --depth 1 https://github.com/gjvarun0307/real-time-moderation-pipeline.git /content/repo

import sys

sys.path.insert(0, "/content/repo/src")

In [ ]:
# install requirements
!pip install -q "transformers>=4.51" "sentencepiece>=0.2" "scikit-learn>=1.5" "pyarrow>=17.0"

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/moderation-pipeline/data/processed")
CHECKPOINT_ROOT = Path("/content/drive/MyDrive/moderation-pipeline/checkpoints")
CHECKPOINT_DIR = CHECKPOINT_ROOT / "teacher-v1"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## Config

In [ ]:
import torch

MODEL_NAME = "xlm-roberta-base"
MAX_SEQ_LEN = 192  # decided from tokenizer fertility (ja p99)
BATCH_SIZE = 128  # A100/H100, so 128 for faster epochs
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
POS_WEIGHT_CAP = 50.0  # caps class-weighted BCE for the rarest label (threat)
GRAD_CLIP_NORM = 1.0
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "no GPU detected — Runtime > Change runtime type > GPU (A100/H100)"
print(f"device: {DEVICE}, {torch.cuda.get_device_name(0)}")

torch.manual_seed(SEED)

## Load prepared data

In [ ]:
import pandas as pd

train_df = pd.read_parquet(DATA_DIR / "train_en.parquet")
val_df = pd.read_parquet(DATA_DIR / "val_en.parquet")
eval_ml_df = pd.read_parquet(DATA_DIR / "eval_multilingual.parquet")

print(f"train_en: {len(train_df)}  val_en: {len(val_df)}  eval_multilingual: {len(eval_ml_df)}")

## Tokenizer, datasets, dataloaders

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from training.data_prep import LABEL_COLUMNS
from training.teacher_model import ToxicityDataset, compute_pos_weight, make_collate_fn

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collate_fn = make_collate_fn(tokenizer, MAX_SEQ_LEN)

train_ds = ToxicityDataset(train_df)
val_ds = ToxicityDataset(val_df)
# eval_multilingual only carries the single "toxic" label
eval_ml_ds = ToxicityDataset(eval_ml_df, label_columns=["toxic"])

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
eval_ml_loader = DataLoader(
    eval_ml_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

pos_weight = compute_pos_weight(train_df, cap=POS_WEIGHT_CAP).to(DEVICE)
pos_weight_by_label = dict(zip(LABEL_COLUMNS, pos_weight.tolist(), strict=True))
print("pos_weight per label:", pos_weight_by_label)

## Model, optimizer, schedule, loss

In [ ]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABEL_COLUMNS), problem_type="multi_label_classification"
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
num_training_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * num_training_steps),
    num_training_steps=num_training_steps,
)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

## Training loop

Raw PyTorch loop, not `Trainer` — the class-weighted BCE loss needs direct access to logits and `pos_weight`, and this same loop shape carries over to the KD loss in `03_student_distill`.

In [ ]:
from training.teacher_model import evaluate

step = 0
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(**batch).logits
            loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        step += 1
        if step % 200 == 0:
            avg_loss = running_loss / 200
            print(f"epoch {epoch + 1} step {step}/{num_training_steps} loss={avg_loss:.4f}")
            running_loss = 0.0

    val_scores = evaluate(model, val_loader, DEVICE)
    print(f"\n=== epoch {epoch + 1} val_en PR-AUC ===")
    for label, score in val_scores.items():
        print(f"  {label}: {score:.4f}")
    print()

## Final evaluation

In [ ]:
final_val_scores = evaluate(model, val_loader, DEVICE)
final_ml_scores = evaluate(model, eval_ml_loader, DEVICE, label_columns=["toxic"])

per_lang_scores = {}
for lang in sorted(eval_ml_df["lang"].unique()):
    lang_df = eval_ml_df[eval_ml_df["lang"] == lang]
    lang_ds = ToxicityDataset(lang_df, label_columns=["toxic"])
    lang_loader = DataLoader(
        lang_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
    )
    per_lang_scores[lang] = evaluate(model, lang_loader, DEVICE, label_columns=["toxic"])

print("val_en (English, held-out):", final_val_scores)
print("eval_multilingual (es+it+tr combined, toxic-only):", final_ml_scores)
print("per-language toxic PR-AUC:", per_lang_scores)

In [ ]:
import json

metrics = {
    "model_name": MODEL_NAME,
    "max_seq_len": MAX_SEQ_LEN,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight_cap": POS_WEIGHT_CAP,
    "val_en_pr_auc": final_val_scores,
    "eval_multilingual_toxic_pr_auc": final_ml_scores,
    "eval_multilingual_toxic_pr_auc_by_lang": per_lang_scores,
}
with open(CHECKPOINT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nwrote {CHECKPOINT_DIR / 'metrics.json'}")

## Save checkpoint

In [ ]:
model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f"saved teacher checkpoint + tokenizer to {CHECKPOINT_DIR}")

## Next

This checkpoint and `metrics.json` are both in `CHECKPOINT_DIR` on Drive.